# Tomorrow's Demand Predictions: Live

Generates next-day district-level predictions using today's live data,
without requiring a full ETL re-run.

**Strategy**: take the last known feature row per district from `features.parquet`,
then patch temporal, weather, and station features with today's actual values.
Lag/rolling features stay as the last known values (stale if ETL hasn't run recently
— accuracy degrades proportionally).

**Requires** (run beforehand):
- `python3 -m src.data.collection.fetch_live`: station snapshot + weather forecast

**Output**: `data/predictions/predictions_latest.parquet`

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from datetime import date
from pathlib import Path

import geopandas as gpd
import holidays
import lightgbm as lgb
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.3f}'.format)

ROOT            = Path('..').resolve()
FEATURES_PATH   = ROOT / 'data' / 'features' / 'features.parquet'
MODEL_PATH      = ROOT / 'models' / 'best_model.txt'
WEATHER_PATH    = ROOT / 'data' / 'processed' / 'weather_daily.parquet'
BEZIRKE_PATH    = ROOT / 'configs' / 'berlin_bezirke.geojson'
SNAPSHOT_DIR    = ROOT / 'bike_data_berlin'
PREDICTIONS_DIR = ROOT / 'data' / 'predictions'

FEATURE_COLS = [
    'district',
    'dow', 'month', 'is_weekend', 'is_holiday', 'is_pre_holiday', 'is_post_holiday',
    'daylight_hours',
    'lag_1d', 'lag_2d', 'lag_7d', 'lag_14d',
    'roll_3d_mean', 'roll_3d_std',
    'roll_7d_mean', 'roll_7d_std',
    'roll_14d_mean', 'roll_14d_std',
    'active_stations',
    'temperature_2m', 'apparent_temperature', 'precipitation',
    'rain', 'snowfall', 'wind_speed_10m', 'cloud_cover', 'relative_humidity_2m',
    'temp_change_1d', 'apparent_temperature_tomorrow', 'precipitation_tomorrow',
    'apparent_temp_x_weekend',
]
LOW_DEMAND_DISTRICTS = ['Marzahn-Hellersdorf', 'Spandau', 'Reinickendorf']
TIMEZONE = 'Europe/Berlin'

today    = pd.Timestamp.now(tz=TIMEZONE).normalize().tz_localize(None)  # today midnight, naive
tomorrow = today + pd.Timedelta(days=1)
print(f'Today: {today.date()}  |  Predicting for: {tomorrow.date()}')

## 1. Load base features and model

In [38]:
features = pd.read_parquet(FEATURES_PATH)
features['date'] = pd.to_datetime(features['date'])

model = lgb.Booster(model_file=str(MODEL_PATH))

# Last known feature row per district (our base to patch)
base = (
    features
    .dropna(subset=FEATURE_COLS)
    .sort_values('date')
    .groupby('district', observed=True)
    .last()
    .reset_index()
)
base = base[~base['district'].isin(LOW_DEMAND_DISTRICTS)].copy()

print(f'Features: {features.date.min().date()} → {features.date.max().date()}')
print(f'Base row date (last known): {base["date"].max().date()}')
print(f'Lag staleness: {(today - base["date"].max()).days} days')
print(f'Districts: {len(base)}')

Features: 2025-01-01 → 2026-05-01
Base row date (last known): 2026-04-30
Lag staleness: 42 days
Districts: 9


## 2. Patch temporal features for today

In [ ]:
berlin_holidays = set(holidays.Germany(state='BE', years=[today.year, tomorrow.year]).keys())

base['date']          = today
base['dow']           = today.dayofweek
base['month']         = today.month
base['is_weekend']    = int(today.dayofweek >= 5)
base['is_holiday']    = int(today.date() in berlin_holidays)
base['is_pre_holiday']  = int((today + pd.Timedelta(days=1)).date() in berlin_holidays)
base['is_post_holiday'] = int((today - pd.Timedelta(days=1)).date() in berlin_holidays)

# Daylight hours: Spencer astronomical formula for Berlin (52.52°N)
doy   = today.dayofyear
B     = 2 * np.pi * (doy - 1) / 365
decl  = (0.006918 - 0.399912 * np.cos(B) + 0.070257 * np.sin(B)
         - 0.006758 * np.cos(2*B) + 0.000907 * np.sin(2*B)
         - 0.002697 * np.cos(3*B) + 0.001480 * np.sin(3*B))
cos_ha = np.clip(-np.tan(np.deg2rad(52.52)) * np.tan(decl), -1, 1)
base['daylight_hours'] = 2 * np.degrees(np.arccos(cos_ha)) / 15

print(f'dow={base["dow"].iloc[0]}  month={base["month"].iloc[0]}  '
      f'is_weekend={base["is_weekend"].iloc[0]}  is_holiday={base["is_holiday"].iloc[0]}  '
      f'is_pre_holiday={base["is_pre_holiday"].iloc[0]}  is_post_holiday={base["is_post_holiday"].iloc[0]}  '
      f'daylight_hours={base["daylight_hours"].iloc[0]:.2f}h')

## 3. Patch weather features

Uses `weather_daily.parquet` which was updated by `fetch_live.py`:
- Today's row: actual observed weather
- Tomorrow's row: forecast (appended by fetch_live)

In [40]:
weather = pd.read_parquet(WEATHER_PATH)
weather['date'] = pd.to_datetime(weather['date'])

today_weather    = weather[weather['date'] == today]
tomorrow_weather = weather[weather['date'] == tomorrow]

print(f'Today weather row    : {len(today_weather)} found')
print(f'Tomorrow weather row : {len(tomorrow_weather)} found')

if today_weather.empty:
    print('WARNING: no weather for today — using last available row as fallback')
    today_weather = weather.sort_values('date').tail(1)

if tomorrow_weather.empty:
    print('WARNING: no forecast for tomorrow — run fetch_live.py first')

print('\nToday weather:')
print(today_weather[['date','temperature_2m','apparent_temperature','precipitation']].to_string(index=False))
print('\nTomorrow forecast:')
print(tomorrow_weather[['date','temperature_2m','apparent_temperature','precipitation']].to_string(index=False))

Today weather row    : 1 found
Tomorrow weather row : 1 found

Today weather:
      date  temperature_2m  apparent_temperature  precipitation
2026-06-11          15.798                13.805          0.000

Tomorrow forecast:
      date  temperature_2m  apparent_temperature  precipitation
2026-06-12          15.231                13.809          0.200


In [41]:
# Patch today's weather columns onto base
weather_cols = [
    'temperature_2m', 'apparent_temperature', 'precipitation',
    'rain', 'snowfall', 'wind_speed_10m', 'cloud_cover', 'relative_humidity_2m',
]
tw = today_weather.iloc[0]
for col in weather_cols:
    base[col] = tw[col]

# temp_change_1d: today vs yesterday (last row in weather before today)
yesterday_weather = weather[weather['date'] < today].sort_values('date').tail(1)
if not yesterday_weather.empty:
    base['temp_change_1d'] = float(tw['temperature_2m']) - float(yesterday_weather['temperature_2m'].iloc[0])

# Tomorrow's forecast features
if not tomorrow_weather.empty:
    base['apparent_temperature_tomorrow'] = float(tomorrow_weather['apparent_temperature'].iloc[0])
    base['precipitation_tomorrow']        = float(tomorrow_weather['precipitation'].iloc[0])

# Recompute interaction
base['apparent_temp_x_weekend'] = base['apparent_temperature'] * base['is_weekend']

print('Weather patch applied.')
print(f'  temp today: {base["temperature_2m"].iloc[0]:.1f}°C  apparent: {base["apparent_temperature"].iloc[0]:.1f}°C')
print(f'  temp_change_1d: {base["temp_change_1d"].iloc[0]:+.1f}°C')
print(f'  apparent_temperature_tomorrow: {base["apparent_temperature_tomorrow"].iloc[0]:.1f}°C')
print(f'  precipitation_tomorrow: {base["precipitation_tomorrow"].iloc[0]:.1f}mm')

Weather patch applied.
  temp today: 15.8°C  apparent: 13.8°C
  temp_change_1d: +3.9°C
  apparent_temperature_tomorrow: 13.8°C
  precipitation_tomorrow: 0.2mm


## 4. Patch active_stations from live snapshot

In [42]:
# Load today's snapshot (written by fetch_live.py)
snapshot_path = SNAPSHOT_DIR / f'live_{today.date()}.parquet'
if not snapshot_path.exists():
    print(f'WARNING: {snapshot_path.name} not found — active_stations will stay stale')
else:
    snapshot = pd.read_parquet(snapshot_path)
    print(f'Loaded snapshot: {len(snapshot)} stations')

    # Spatially join stations to districts
    bezirke = gpd.read_file(BEZIRKE_PATH).to_crs('EPSG:4326')
    print('Bezirke columns:', bezirke.columns.tolist())
    bezirke.head(3)

Loaded snapshot: 1706 stations
Bezirke columns: ['name', 'description', 'cartodb_id', 'created_at', 'updated_at', 'geometry']


In [43]:
# Identify the district name column from the GeoJSON (adjust if needed after cell above)
DISTRICT_COL = 'name'   # <-- update if bezirke.columns shows a different name

stations_gdf = gpd.GeoDataFrame(
    snapshot,
    geometry=gpd.points_from_xy(snapshot['longitude'], snapshot['latitude']),
    crs='EPSG:4326',
)

# Rename bezirke district column to 'district' before join to avoid collision
# with the station 'name' column (sjoin would otherwise suffix it to 'name_right')
bezirke_join = bezirke[[DISTRICT_COL, 'geometry']].rename(columns={DISTRICT_COL: 'district'})
joined = gpd.sjoin(stations_gdf, bezirke_join, predicate='within', how='left')

live_active = (
    joined.dropna(subset=['district'])
    .groupby('district')
    .size()
    .reset_index(name='active_stations_live')
)
print('Live active stations per district:')
print(live_active.sort_values('active_stations_live', ascending=False).to_string(index=False))

Live active stations per district:
                  district  active_stations_live
                     Mitte                   525
  Friedrichshain-Kreuzberg                   321
Charlottenburg-Wilmersdorf                   178
      Tempelhof-Schöneberg                   137
                    Pankow                   127
               Lichtenberg                   116
          Treptow-Köpenick                    94
                  Neukölln                    86
       Steglitz-Zehlendorf                    75
             Reinickendorf                    26
       Marzahn-Hellersdorf                    17
                   Spandau                     3


In [44]:
# Merge live active_stations into base and compare vs stale values
base = base.merge(live_active, on='district', how='left')
base['active_stations_stale'] = base['active_stations']
base['active_stations'] = base['active_stations_live'].fillna(base['active_stations_stale'])

print('active_stations — stale vs live:')
print(base[['district', 'active_stations_stale', 'active_stations_live', 'active_stations']]
      .sort_values('active_stations', ascending=False).to_string(index=False))

active_stations — stale vs live:
                  district  active_stations_stale  active_stations_live  active_stations
                     Mitte                    530                   525              525
  Friedrichshain-Kreuzberg                    354                   321              321
Charlottenburg-Wilmersdorf                    215                   178              178
      Tempelhof-Schöneberg                    196                   137              137
                    Pankow                    172                   127              127
               Lichtenberg                    190                   116              116
          Treptow-Köpenick                    147                    94               94
                  Neukölln                    148                    86               86
       Steglitz-Zehlendorf                     74                    75               75


## 5. Generate predictions

In [45]:
# Restore categorical dtype to match what the model was trained on
train_districts = (
    features[~features['district'].isin(LOW_DEMAND_DISTRICTS)]['district']
    .cat.remove_unused_categories()
    .cat.categories
)
base['district'] = pd.Categorical(base['district'], categories=train_districts)

base['pred_relative_demand'] = model.predict(base[FEATURE_COLS])
base['pred_rentals']         = base['pred_relative_demand'] * base['active_stations']

print(f'Predictions for {tomorrow.date()}:')
base[['district', 'pred_relative_demand', 'pred_rentals', 'active_stations']].sort_values(
    'pred_rentals', ascending=False
).reset_index(drop=True)

Predictions for 2026-06-12:


,district,pred_relative_demand,pred_rentals,active_stations
0,Mitte,4.699,2467.042,525
1,Friedrichshain-Kreuzberg,6.112,1962.082,321
2,Neukölln,5.396,464.018,86
3,Charlottenburg-Wilmersdorf,2.542,452.392,178
4,Pankow,2.981,378.628,127
5,Tempelhof-Schöneberg,2.483,340.200,137
6,Lichtenberg,2.842,329.722,116
7,Treptow-Köpenick,3.041,285.832,94
8,Steglitz-Zehlendorf,1.469,110.173,75


In [46]:
# Key context: what's driving these predictions?
context_cols = [
    'district', 'dow', 'is_weekend', 'is_holiday', 'daylight_hours',
    'apparent_temperature', 'apparent_temperature_tomorrow',
    'precipitation_tomorrow', 'lag_7d', 'roll_7d_mean',
    'pred_relative_demand', 'pred_rentals',
]
base[context_cols].sort_values('pred_rentals', ascending=False).reset_index(drop=True)

,district,dow,is_weekend,is_holiday,daylight_hours,apparent_temperature,apparent_temperature_tomorrow,precipitation_tomorrow,lag_7d,roll_7d_mean,pred_relative_demand,pred_rentals
0,Mitte,3,0,0,16.491,13.805,13.809,0.200,4.279,4.134,4.699,2467.042
1,Friedrichshain-Kreuzberg,3,0,0,16.491,13.805,13.809,0.200,5.261,5.355,6.112,1962.082
2,Neukölln,3,0,0,16.491,13.805,13.809,0.200,4.310,4.695,5.396,464.018
3,Charlottenburg-Wilmersdorf,3,0,0,16.491,13.805,13.809,0.200,2.230,2.180,2.542,452.392
4,Pankow,3,0,0,16.491,13.805,13.809,0.200,2.587,2.753,2.981,378.628
5,Tempelhof-Schöneberg,3,0,0,16.491,13.805,13.809,0.200,2.119,2.187,2.483,340.200
6,Lichtenberg,3,0,0,16.491,13.805,13.809,0.200,2.146,2.510,2.842,329.722
7,Treptow-Köpenick,3,0,0,16.491,13.805,13.809,0.200,3.039,2.952,3.041,285.832
8,Steglitz-Zehlendorf,3,0,0,16.491,13.805,13.809,0.200,1.486,1.471,1.469,110.173


## 6. Save

In [47]:
predictions = base[[
    'district', 'active_stations',
    'apparent_temperature_tomorrow', 'precipitation_tomorrow',
    'pred_relative_demand', 'pred_rentals',
]].copy()
predictions.insert(0, 'prediction_date', tomorrow)
predictions.insert(1, 'features_date',   today)
predictions = predictions.sort_values('pred_rentals', ascending=False).reset_index(drop=True)

PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)
latest_path = PREDICTIONS_DIR / 'predictions_latest.parquet'
dated_path  = PREDICTIONS_DIR / f'predictions_{tomorrow.date()}.parquet'

predictions.to_parquet(latest_path, index=False)
predictions.to_parquet(dated_path,  index=False)

print(f'Saved → {latest_path.name}')
print(f'Saved → {dated_path.name}')
print(f'\nFinal predictions:')
print(predictions.to_string(index=False))

Saved → predictions_latest.parquet
Saved → predictions_2026-06-12.parquet

Final predictions:
prediction_date features_date                   district  active_stations  apparent_temperature_tomorrow  precipitation_tomorrow  pred_relative_demand  pred_rentals
     2026-06-12    2026-06-11                      Mitte              525                         13.809                   0.200                 4.699      2467.042
     2026-06-12    2026-06-11   Friedrichshain-Kreuzberg              321                         13.809                   0.200                 6.112      1962.082
     2026-06-12    2026-06-11                   Neukölln               86                         13.809                   0.200                 5.396       464.018
     2026-06-12    2026-06-11 Charlottenburg-Wilmersdorf              178                         13.809                   0.200                 2.542       452.392
     2026-06-12    2026-06-11                     Pankow              127        